# Notebook 3 — Designing an Automated Trading System

Notebook 1 found a rule. Notebook 2 proved the rule can drive real orders against the exchange. Both were deliberately minimal — a 30-line loop, in-memory state, market orders, no error handling beyond a `print`.

That's fine for a 3-minute live demo. It is not what you'd point at a real trading session. This notebook is about the gap between the two: what a naive signal loop quietly gets away with, and what an actual automated trading system (ATS) has to handle instead. No execution here — this is a design pass. `spread-trader.py` is where the design below gets built.

## What notebook 2 gets away with

Every one of these is fine for a supervised demo and wrong for anything unattended:

| notebook 2 | why it's a problem unattended |
|---|---|
| Sequential HTTP calls, one leg at a time | latency scales with the number of legs — by the time leg 2's order lands, the price it was sized against may have moved |
| Assumes it starts flat (`last_position = 0`) | a restart after a crash mid-position either fights the position it already has, or silently forgets it's exposed |
| Rejected orders just get `print`ed | a `MAX_POSITION` breach or a `429` is treated the same as success — the loop moves on with a position it thinks it has and doesn't |
| Market orders only | pays the taker fee and crosses the spread on *every* rebalance, unconditionally |
| Log is a Python list in memory | a crash loses the entire run — nothing to debug after the fact |
| Fixed `time.sleep`, no backoff | a slow tick or a rate-limited response just makes the next tick late — no signal that anything's wrong |
| No kill switch | if the signal itself misbehaves (bad data, flip-flopping every tick), nothing stops it from trading forever |

None of this is a criticism of notebook 2 — it did its one job, which was proving the wiring works. This is the list of things you'd get burned by if you just left it running.

## Architecture

Split into layers with one job each, so a bug in one (a bad signal, say) can't silently corrupt another (risk limits still apply no matter what the signal asks for):

```mermaid
flowchart TB
    MD[Market Data Layer<br/>concurrent GET /products per tick]
    SIG[Signal Layer<br/>rolling z-score, hysteresis<br/>pure function, no I/O]
    RISK[Risk Layer<br/>clamp to MAX_POSITION headroom<br/>kill switch on repeated rejects]
    EXEC[Execution Layer<br/>reconcile-by-diff<br/>concurrent order submission<br/>429 retry with backoff]
    API[(Exchange API)]
    LOG[(Structured log— CSV, flushed every tick)]

    MD --> SIG --> RISK --> EXEC --> API
    MD -.-> LOG
    SIG -.-> LOG
    EXEC -.-> LOG
    API -.->|account, fills| MD
```

The signal layer talks to nothing external — it's the exact same class from notebook 2, unchanged, so research and production share one decision function instead of two implementations that can quietly drift apart.

## Layer-by-layer design decisions

**Market data.** Fetch every leg's index price concurrently (one thread-pool submission per product, like `arb.py` does for its book pulls) instead of sequentially — a 2-leg tick becomes one round-trip's worth of wall-clock time instead of two.

**Signal.** Unchanged from notebooks 1–2: rolling mean/std over `WINDOW` samples, entry at ±`ENTRY_Z`, flatten below `EXIT_Z`. Kept as a class with no network calls in it, so it can be unit-tested against a list of floats without touching the exchange.

**Risk.** Sits between the signal and execution, and it is never allowed to be the signal's problem to enforce:
- clamp the requested target position to whatever headroom is actually left under each product's `max_position` (pulled live from `GET /products`, not hardcoded — same principle notebook 1's data generator used, don't re-implement server config on the client)
- count consecutive order rejections; past a threshold, trip a kill switch — stop trading and flatten, rather than hammering an exchange that keeps saying no
- refuse to act on a price that looks stale or missing rather than trusting every response blindly

**Execution.** Reconciliation is still "submit the diff between target and current," exactly like notebook 2 — but now: both legs submit concurrently (neither leg sits exposed waiting on the other), and any call that comes back `429` retries with exponential backoff instead of treating a rate limit as a hard failure.

**Persistence.** Every tick — not just every trade — gets written to a CSV immediately and flushed, not buffered in a Python list. A crash mid-run loses at most the row in flight, and the full run is inspectable afterward the same way notebook 2's live chart was, except it survives the process dying.

**Orchestration.** On startup, read actual account positions from `GET /account` and use that as the starting state — never assume flat. On `SIGINT`/`SIGTERM`, stop taking new signals, flatten every open leg, and only then exit. That turns "someone hit Ctrl+C" from an unmanaged position into the same clean shutdown notebook 2 did by hand at the end.

## Config surface

| knob | notebook 2 | spread-trader.py | why it moved |
|---|---|---|---|
| `MAX_POSITION` per leg | assumed, never checked | read live from `GET /products` | server is the source of truth; a config drift shouldn't cause silent rejections |
| retry on `429` | none | exponential backoff, capped retries | matches `arb.py`'s shared-session pattern — expected under concurrent I/O, not fatal |
| kill switch | none | trips after N consecutive rejections | stops a misbehaving run instead of trading blind |
| startup state | assumed flat | read from `GET /account` | correctness after a restart, not just at cold start |
| logging | in-memory list | CSV, flushed per tick | survives a crash |
| shutdown | manual "flatten" cell | signal handler, automatic | same hygiene, not opt-in |

## Deliberately out of scope

Worth naming so it's a decision, not an oversight:
- **Still market orders.** Limit-order execution (post at a level, earn the maker rebate, fall back to marketable if unfilled after some time) would cut the transaction-cost drag notebook 2's equity curve showed, but it's a materially bigger execution state machine. Good stretch goal, not this workshop.
- **Fixed sizing.** `TARGET_QTY` doesn't scale with how extreme `z` is. A conviction-weighted size (e.g. proportional to `min(|z|, cap)`) is a straightforward follow-on once the base loop is trusted.
- **Single strategy, two legs.** No portfolio layer, no multiple concurrent signals sharing one risk budget. Everything here is scoped to exactly the BTC-MINI/ETH-MINI pair from notebooks 1–2.
- **No persistent database.** The CSV log is sufficient for a workshop-length run; a real system would want a durable store it can query concurrently while still trading.

`spread-trader.py` implements everything above the line, not everything in this section.